# 04 -- Validation (purged CV + walk-forward)

Validate the signal out-of-sample with **purged k-fold** and **walk-forward** splits that respect label spans and embargo.

In [ ]:
# Parameters (papermill-overridable: `papermill ... -p SYMBOLS '["AAPL","MSFT"]'`)
SYMBOLS = ["ALPHA", "BRAVO", "CHARLIE"]
START = "2018-01-01"
N_DAYS = 600
SEED = 7
USE_SYNTHETIC = True  # set False to fetch real data via core_trading.data.sources


In [ ]:
import numpy as np
import pandas as pd

from core_trading.research.reproducibility import set_seeds

set_seeds(SEED)


def synthetic_bars(symbols, n, seed, start=START):
    """Seeded OHLCV frame in the canonical (symbol, timestamp) layout."""
    rng = np.random.default_rng(seed)
    idx = pd.date_range(start, periods=n, freq="B", tz="UTC")
    frames = []
    for k, sym in enumerate(symbols):
        drift = 0.0003 * (1 + k)
        px = 100.0 + np.cumsum(rng.standard_normal(n) + drift)
        px = np.maximum(px, 1.0)
        high = px + np.abs(rng.standard_normal(n)) * 0.4
        low = px - np.abs(rng.standard_normal(n)) * 0.4
        frame = pd.DataFrame(
            {
                "open": px,
                "high": np.maximum(high, px),
                "low": np.minimum(low, px),
                "close": px,
                "volume": rng.uniform(1e6, 5e6, n),
                "source": "synthetic",
            },
            index=pd.MultiIndex.from_product(
                [[sym], idx], names=["symbol", "timestamp"]
            ),
        )
        frames.append(frame)
    return pd.concat(frames).sort_index()


if USE_SYNTHETIC:
    bars = synthetic_bars(SYMBOLS, N_DAYS, SEED)
else:  # pragma: no cover - exercised only against live vendors
    import asyncio

    from core_trading.data.bars import BarRequest, BarResolution
    from core_trading.data.sources.yfinance_source import YFinanceBarSource

    req = BarRequest(
        symbols=tuple(SYMBOLS),
        resolution=BarResolution.DAY_1,
        start=pd.Timestamp(START, tz="UTC").to_pydatetime(),
        end=pd.Timestamp.now(tz="UTC").to_pydatetime(),
    )
    bars = asyncio.run(YFinanceBarSource().fetch_bars(req))

print(f"loaded {bars.shape[0]} bars across {len(SYMBOLS)} symbols")
bars.head()


In [ ]:
from core_trading.research.feature_store import default_feature_store
from core_trading.research.cross_validation import (
    PurgedKFold, WalkForwardSplit, make_label_end_times)

store = default_feature_store()
z = store.compute(bars, ['zscore_20'])['zscore_20']
signal = (-z / 2.0).clip(-1.0, 1.0)
fwd = bars['close'].groupby(level='symbol').pct_change().groupby(
    level='symbol').shift(-1)

# Work on a single symbol for the CV illustration.
sym = SYMBOLS[0]
s = signal.xs(sym, level='symbol')
r = fwd.xs(sym, level='symbol')
panel = pd.concat([s.rename('sig'), r.rename('ret')], axis=1).dropna()
times = panel.index

In [ ]:
def fold_ic(idx):
    sub = panel.iloc[idx]
    return sub['sig'].corr(sub['ret'])

t1 = make_label_end_times(times, 1)
pkf = PurgedKFold(n_splits=5, embargo_pct=0.02)
oos_ic = [fold_ic(sp.test_indices) for sp in pkf.split(times, t1)]
print('purged k-fold OOS ICs:', [round(x, 4) for x in oos_ic])
print('mean OOS IC:', round(float(np.nanmean(oos_ic)), 4))

In [ ]:
wf = WalkForwardSplit(n_splits=5, test_size=60, anchored=True)
wf_ic = [fold_ic(sp.test_indices) for sp in wf.split(times)]
print('walk-forward OOS ICs:', [round(x, 4) for x in wf_ic])

**Gate:** OOS IC stable and same-signed across folds -> proceed to `05_portfolio.ipynb`. Unstable signs across folds = overfit; stop.